# 🎮 Notebook 04 — Playstyle Clustering

**Phase 6 du projet KCorp Scouting Tool**

Ce notebook documente l'analyse non-supervisée des styles de jeu des joueurs ERL.
Il retrace le **raisonnement itératif** : pourquoi DBSCAN a échoué, pourquoi on a pivoté
vers un clustering par position, et ce que les résultats nous disent.

## Sommaire
1. [Contexte & Objectif](#1-contexte)
2. [Chargement et préparation](#2-data)
3. [Tentative 1 : K-Means global (toutes positions)](#3-kmeans-global)
4. [Tentative 2 : DBSCAN — Échec documenté](#4-dbscan)
5. [Solution finale : Clustering par position](#5-par-position)
6. [Visualisations UMAP](#6-umap)
7. [Profils des clusters](#7-profils)
8. [Similarity Search — "Trouve le prochain X"](#8-similarity)
9. [Conclusions](#9-conclusions)

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

from IPython.display import display
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

ROOT = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path('.').resolve()
METRICS_DIR = ROOT / 'reports' / 'metrics'
FIGURES_DIR = ROOT / 'reports' / 'figures'

plt.style.use('dark_background')
plt.rcParams.update({'font.family': 'DejaVu Sans', 'figure.facecolor': '#0f172a',
                     'axes.facecolor': '#1e293b', 'text.color': '#f1f5f9'})

print('Imports OK')

---
## 1. Contexte & Objectif <a id='1-contexte'></a>

### Pourquoi le clustering ?

Le **Talent Score** (Notebook 03) répond à : *"Ce joueur va-t-il être promu ?"*  
Le **Clustering** répond à : *"Comment joue-t-il ?"*

Ces deux dimensions sont complémentaires pour un scout :
- Un club cherche un joueur avec un **style précis** (carry agressif vs. playmaker)
- Identifier les styles "LEC-ready" aide à comprendre *pourquoi* certains joueurs progressent

### Features utilisées

On utilise les **9 Z-scores** (relatifs à la ligue et à la position), pas les valeurs brutes :
- Les Z-scores capturent le *style* indépendamment du *niveau* de la ligue
- Un Z-score DPM = +1.5 signifie "ce joueur fait beaucoup plus de dégâts que la moyenne de sa ligue"

---
## 2. Chargement et préparation <a id='2-data'></a>

In [ ]:
from src.models.clusterer import load_data, CLUSTER_FEATURES, ERL_LEAGUES, POSITIONS

df = load_data()
available_features = [f for f in CLUSTER_FEATURES if f in df.columns]

print(f'Dataset ERL : {len(df):,} joueurs/splits')
print(f'Ligues : {df["league"].unique().tolist()}')
print(f'Positions : {df["position"].value_counts().to_dict()}')
print(f'\nFeatures ({len(available_features)}) :')
for f in available_features:
    print(f'  {f}')

---
## 3. Tentative 1 : K-Means Global <a id='3-kmeans-global'></a>

On commence par le cas le plus simple : un seul modèle K-Means sur tous les joueurs ERL,
toutes positions confondues. La méthode du coude + silhouette score pour choisir k.

In [ ]:
X_all = df[available_features].fillna(0).values
scaler_global = StandardScaler()
X_sc = scaler_global.fit_transform(X_all)

k_range = range(3, 10)
inertias, silhouettes = [], []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_sc)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_sc, labels)
    silhouettes.append(sil)
    print(f'k={k} | Inertie={km.inertia_:,.0f} | Silhouette={sil:.4f}')

best_k_global = list(k_range)[np.argmax(silhouettes)]
print(f'\nMeilleur k (global) : {best_k_global}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('K-Means Global — Sélection du k', fontweight='bold')

ax = axes[0]
ax.set_facecolor('#1e293b')
ax.plot(k_range, inertias, 'o-', color='#60a5fa', lw=2)
ax.axvline(best_k_global, color='#facc15', ls='--', lw=1.5, label=f'k={best_k_global}')
ax.set_title('Méthode du coude (Inertie)')
ax.set_xlabel('k')
ax.set_ylabel('Inertie')
ax.legend()

ax = axes[1]
ax.set_facecolor('#1e293b')
ax.plot(k_range, silhouettes, 's-', color='#34d399', lw=2)
ax.scatter([best_k_global], [max(silhouettes)], color='#facc15', s=100, zorder=5)
ax.set_title('Silhouette Score')
ax.set_xlabel('k')
ax.set_ylabel('Silhouette')

plt.tight_layout()
plt.show()
print(f'\nObservation : k=3 donne le meilleur silhouette ({max(silhouettes):.4f}) mais c\'est faible.')
print('Les clusters globaux mélangent des rôles très différents (ADC vs Support).')

---
## 4. Tentative 2 : DBSCAN — Échec documenté <a id='4-dbscan'></a>

### Qu'est-ce que DBSCAN ?

DBSCAN (Density-Based Spatial Clustering) forme des clusters en identifiant des **régions denses**.
Il ne nécessite pas de spécifier k et peut détecter des outliers (label = -1).

### Pourquoi ça échoue ici

La **malédiction de la dimensionnalité** (*curse of dimensionality*) : en espace de haute dimension
(9 features), les distances euclidiennes entre tous les points tendent à converger vers la même valeur.
La notion de "densité locale" perd son sens — DBSCAN ne peut plus distinguer les voisins proches
des voisins éloignés.

In [ ]:
from sklearn.cluster import DBSCAN

print('Test DBSCAN sur plusieurs valeurs de eps :')
print(f'{"eps":<8} {"clusters":<12} {"bruit":<12} {"silhouette"}')
print('-' * 50)

for eps in [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0]:
    db = DBSCAN(eps=eps, min_samples=8)
    labels_db = db.fit_predict(X_sc)
    n_clusters = len(set(labels_db)) - (1 if -1 in labels_db else 0)
    n_noise = (labels_db == -1).sum()
    noise_pct = n_noise / len(labels_db)
    
    if n_clusters > 1:
        sil = silhouette_score(X_sc, labels_db)
        sil_str = f'{sil:.4f}'
    else:
        sil_str = 'N/A'
    
    print(f'{eps:<8} {n_clusters:<12} {noise_pct:<12.1%} {sil_str}')

print('\n=> Conclusion : eps petit = 100% bruit. eps grand = 1 mega-cluster.')
print('   DBSCAN ne fonctionne pas sur des données tabulaires normalisées en 9D.')
print('   Decision : abandon de DBSCAN, pivot vers K-Means par position.')

> **Leçon de Data Science** : DBSCAN est très efficace en 2D/3D sur des données géographiques
> ou avec des densités très hétérogènes. Sur des données tabulaires à 9 dimensions normalisées,
> K-Means est systématiquement plus robuste.

---
## 5. Solution finale : Clustering par position <a id='5-par-position'></a>

**Insight** : Un support avec un bon DPM n'est pas comparable à un ADC avec un bon DPM.
Les 5 rôles ont des profils statistiques fondamentalement différents.

→ On entraîne **5 modèles K-Means indépendants**, un par position.

In [ ]:
# Résumé des résultats par position (déjà calculé)
summary = pd.read_csv(METRICS_DIR / 'clustering_summary.csv')
display(summary.round(4))

In [ ]:
# Taux de promotion par cluster et par position
with open(METRICS_DIR / 'cluster_profiles.json', encoding='utf-8') as f:
    profiles = json.load(f)

print('Taux de promotion par cluster (position par position) :')
print(f'{"Position":<10} {"Cluster":<10} {"Archétype":<45} {"Joueurs":<10} {"Promos"}')
print('-' * 100)
for pos, pos_profiles in profiles.items():
    for p in sorted(pos_profiles, key=lambda x: -x['promo_rate']):
        print(f'{pos:<10} C{p["cluster"]:<9} {p["archetype"]:<45} {p["n_players"]:<10} {p["promo_rate"]:.1%}')

---
## 6. Visualisations UMAP <a id='6-umap'></a>

UMAP réduit les 9 dimensions de nos features en 2D pour la visualisation.
Les **étoiles** = joueurs promus en LEC.

In [ ]:
from IPython.display import Image, display as ipy_display
ipy_display(Image(str(FIGURES_DIR / '07_umap_by_position.png'), width=1100))

In [ ]:
print('Courbes du coude et silhouette par position :')
ipy_display(Image(str(FIGURES_DIR / '09_elbow_silhouette.png'), width=1100))

---
## 7. Profils des clusters <a id='7-profils'></a>

In [ ]:
from IPython.display import Image, display as ipy_display
print('Heatmap des profils moyens par cluster :')
ipy_display(Image(str(FIGURES_DIR / '08_cluster_profiles_heatmap.png'), width=1200))

### Lecture de la heatmap

- **Vert** = Z-score positif (au-dessus de la moyenne de la ligue pour cette feature)
- **Rouge** = Z-score négatif (en-dessous)
- **Le % à droite du label de ligne** = taux de promotion LEC de ce cluster

**Pattern universel** : le cluster avec le taux de promotion le plus élevé est toujours
celui avec les Z-scores positifs sur DPM, Gold@15, XP@15 → *le joueur dominant sa ligue
en early game a plus de chances d'être promu.*

---
## 8. Similarity Search — "Trouve le prochain X" <a id='8-similarity'></a>

Pour chaque position, on peut trouver les joueurs ERL dont le style statistique
est le plus proche d'un joueur cible (par distance euclidienne dans l'espace des Z-scores).

In [ ]:
from src.models.clusterer import find_similar_players, CLUSTER_FEATURES

df_clustering = pd.read_csv(METRICS_DIR / 'clustering_results.csv')
available = [f for f in CLUSTER_FEATURES if f in df_clustering.columns]

# Normalisation globale pour la similarity search
scaler_sim = StandardScaler()
X_sim = scaler_sim.fit_transform(df_clustering[available].fillna(0))

# Exemple : joueurs similaires à zoelys (top scorer LFL Support)
print('Joueurs ERL dont le style ressemble à zoelys (LFL · Support) :')
similar = find_similar_players('zoelys', df_clustering, X_sim, top_k=8)
if not similar.empty:
    display(similar)

In [ ]:
# Joueurs similaires à un autre top scoreur
print('Joueurs ERL dont le style ressemble à shelfmade (PRM · Top) :')
similar2 = find_similar_players('shelfmade', df_clustering, X_sim, top_k=8)
if not similar2.empty:
    display(similar2)

---
## 9. Conclusions <a id='9-conclusions'></a>

### Ce qu'on a appris

1. **DBSCAN échoue** en espace de haute dimension → documenté comme exemple pédagogique

2. **K-Means global** donne des clusters trop hétérogènes (mix de rôles incomparables)

3. **K-Means par position** (k=3 pour toutes les positions) révèle un pattern universel :
   - Cluster "dominant early game" : **15-23% de taux de promotion** selon la position
   - Cluster "profil neutre" : **1-6%** — ces joueurs ont moins de chances d'être promus

4. **La similarity search** permet une utilisation concrète par un scout : 
   *"Trouve-moi un ADC qui joue comme [star LEC]"*

### Synergies Talent Score × Clustering

| Question du scout | Outil |
|---|---|
| Qui va progresser ? | **Talent Score** (Notebook 03) |
| Comment joue-t-il ? | **Clustering** (ce notebook) |
| Qui joue comme [star] ? | **Similarity Search** |
| Quel style domine dans quelle ligue ? | **UMAP par position** |

→ **Notebook 05** : Synthèse finale et top recommandations